In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, AutoModelForCausalLM
import torch

In [2]:
# The Document (Longer than 100 words to make summary)
document = """
Artificial Intelligence (AI) is a field that focuses on creating machines capable of intelligent behavior.
Machine Learning (ML) is a subset of AI that enables machines to learn from data.
Deep Learning (DL) uses neural networks with multiple layers for complex pattern recognition.
Applications of AI include natural language processing, computer vision, robotics, healthcare, finance, and recommendation systems.
Ethical AI ensures fairness, accountability, and transparency in AI deployments.
AI continues to evolve rapidly, impacting numerous industries and research domains.
Modern AI systems are being integrated into smartphones, autonomous vehicles, and smart homes.
The history of AI dates back to the 1950s with pioneers like Alan Turing.
The future of AI holds potential for General Intelligence, where machines can perform any intellectual task a human can.
Challenges in AI include data privacy, algorithmic bias, and high energy consumption for training large models.
However, the benefits in medical diagnosis and scientific discovery are immense.
"""

In [3]:
# Force embedding to CPU to prevent CUDA memory crashes
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [4]:
#  Setup Models
print("Loading 'real' models... (this may take a minute)")

Loading 'real' models... (this may take a minute)


In [5]:
# Embedding model: Standard industry choice for sentence similarity
embedder = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
# Summarization model: Using BART (specialized for summarizing)
sum_name = "facebook/bart-large-cnn"
sum_tokenizer = AutoTokenizer.from_pretrained(sum_name)
sum_model = AutoModelForSeq2SeqLM.from_pretrained(sum_name)

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [7]:
# LLM for QA: DistilGPT2 (Lightweight local LLM)
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [8]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [9]:
# Chunking (30 words)
def chunk_text(text, chunk_size=30):
    words = text.split()
    return [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]

In [10]:
# Workflow Logic
def run_real_qa(query, text):
    # Chunking
    chunks = chunk_text(text)

    # Embeddings
    chunk_embeddings = embedder.encode(chunks)
    query_embedding = embedder.encode([query])

    # Cosine Similarity
    # cosine_similarity returns a matrix; [0] gets scores for our single query
    similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]

    # Retrieve Top 2
    top_indices = np.argsort(similarities)[-2:][::-1]
    relevant_chunks = [chunks[i] for i in top_indices]
    context_text = " ".join(relevant_chunks)

    # Summarization (if > 100 words)
    summary_text = ""
    if len(text.split()) > 100:
        # Summarize the entire doc to provide high-level context
        inputs = sum_tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)
        summary_ids = sum_model.generate(inputs["input_ids"], max_length=50, min_length=10, do_sample=False)
        summary_text = sum_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        summary_text = f"Summary: {summary_text}\n"

    # Real Generation
    prompt = f"{summary_text}Relevant Context: {context_text}\n\nQuestion: {query}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.3, # Low temperature for "real" factual answers
        do_sample=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [11]:
# Execution
query = "What are the applications and ethical concerns of AI?"
answer = run_real_qa(query, document)
print("\n--- FINAL OUTPUT ---")
print(answer)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



--- FINAL OUTPUT ---
Summary: Artificial Intelligence (AI) is a form of machine learning. It can be used to make decisions based on patterns in data.
Relevant Context: Deep Learning (DL) uses neural networks with multiple layers for complex pattern recognition. Applications of AI include natural language processing, computer vision, robotics, healthcare, finance, and recommendation systems. Ethical AI ensures fairness, accountability, and transparency in AI deployments. AI continues to evolve rapidly, impacting numerous industries and research domains. Modern AI systems are being integrated into smartphones, autonomous vehicles, and

Question: What are the applications and ethical concerns of AI?
Answer: Artificial Intelligence is a form of machine learning. It can be used to make decisions based on patterns in data.
Question: What are the applications and ethical concerns of AI?
Answer: Artificial Intelligence is a form of machine learning. It can be
